# GitHub PR Review

Thin notebook wrapper for the PR-review LangGraph implementation in `github_review_util.py`.

In [ ]:
from pathlib import Path
import sys

import gradio as gr

util_candidates = [
    Path.cwd(),
    Path.cwd() / "4_langgraph" / "community_contributions" / "misi",
]

for candidate in util_candidates:
    if (candidate / "github_review_util.py").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

from github_review_util import (
    build_review_graph,
    default_model,
    default_repository,
    load_review_environment,
    new_thread_config,
)

load_review_environment()
graph = build_review_graph()
config = new_thread_config()

print(f"Default repository: {default_repository() or '(not set)'}")
print(f"Review model: {default_model()}")

In [ ]:
async def chat(user_input, history):
    result = await graph.ainvoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config,
    )
    return result.get("review") or result["messages"][-1].content


gr.ChatInterface(
    chat,
    type="messages",
    title="GitHub PR Reviewer",
    examples=[
        "Review PR 1",
        "Review mihaly-hazag/test-repo#1",
        "Review https://github.com/mihaly-hazag/test-repo/pull/1",
    ],
).launch()